In [1]:
import pandas as pd
import sqlite3

In [3]:
orders=pd.read_csv("orders.csv")
users=pd.read_json("users.json")

orders.head(),users.head()


(   order_id  user_id  restaurant_id  order_date  total_amount  \
 0         1     2508            450  18-02-2023        842.97   
 1         2     2693            309  18-01-2023        546.68   
 2         3     2084            107  15-07-2023        163.93   
 3         4      319            224  04-10-2023       1155.97   
 4         5     1064            293  25-12-2023       1321.91   
 
                   restaurant_name  
 0               New Foods Chinese  
 1  Ruchi Curry House Multicuisine  
 2           Spice Kitchen Punjabi  
 3          Darbar Kitchen Non-Veg  
 4       Royal Eatery South Indian  ,
    user_id    name       city membership
 0        1  User_1    Chennai    Regular
 1        2  User_2       Pune       Gold
 2        3  User_3  Bangalore       Gold
 3        4  User_4  Bangalore    Regular
 4        5  User_5       Pune       Gold)

In [5]:
conn = sqlite3.connect(":memory:")

with open("restaurants.sql","r",
          encoding="utf-8")as f:
    sql_script = f.read()

conn.executescript(sql_script)

restaurants=pd.read_sql("SELECT * FROM restaurants",conn)
restaurants.head()


,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [6]:
df = orders.merge(users, on="user_id", how="left")
df = df.merge(restaurants, on="restaurant_id", how="left")

df.head()

,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [7]:
df.to_csv("final_food_delivery_dataset.csv", index=False)
print("✅ final_food_delivery_dataset.csv created")

✅ final_food_delivery_dataset.csv created


In [8]:
df[df["membership"]=="Gold"] \
.groupby("city")["total_amount"].sum() \
.sort_values(ascending=False)

city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

In [9]:
df.groupby("cuisine")["total_amount"] \
.mean() \
.sort_values(ascending=False)

cuisine
Mexican    808.021344
Italian    799.448578
Indian     798.466011
Chinese    798.389020
Name: total_amount, dtype: float64

In [10]:
user_total = df.groupby("user_id")["total_amount"].sum()

(user_total > 1000).sum()

np.int64(2544)

In [11]:
bins = [3.0, 3.5, 4.0, 4.5, 5.0]
labels = ["3.0 – 3.5", "3.6 – 4.0", "4.1 – 4.5", "4.6 – 5.0"]

df["rating_range"] = pd.cut(df["rating"], bins=bins, labels=labels)

df.groupby("rating_range")["total_amount"] \
.sum() \
.sort_values(ascending=False)

rating_range
4.6 – 5.0    2197030.75
4.1 – 4.5    1960326.26
3.0 – 3.5    1881754.57
3.6 – 4.0    1717494.41
Name: total_amount, dtype: float64

In [12]:
df[df["membership"] == "Gold"] \
.groupby("city")["total_amount"] \
.mean() \
.sort_values(ascending=False)

city
Chennai      808.459080
Hyderabad    806.421034
Bangalore    793.223756
Pune         781.162243
Name: total_amount, dtype: float64

In [13]:
df.groupby("cuisine").agg(
    restaurant_count=("restaurant_id", "nunique"),
    total_revenue=("total_amount", "sum")
).sort_values("restaurant_count")

,restaurant_count,total_revenue
cuisine,,
Chinese,120,1930504.65
Indian,126,1971412.58
Italian,126,2024203.80
Mexican,128,2085503.09


In [14]:
gold_percentage = (df["membership"] == "Gold").mean() * 100
round(gold_percentage)

50

In [19]:
df.groupby("restaurant_name_x").agg(
    total_orders=("order_id", "count"),
    avg_order_value=("total_amount", "mean")
).query("total_orders < 20") \
.sort_values("avg_order_value", ascending=False)

,total_orders,avg_order_value
restaurant_name_x,,
Hotel Dhaba Multicuisine,13,1040.222308
Sri Mess Punjabi,12,1029.180833
Ruchi Biryani Punjabi,16,1002.140625
Sri Delights Pure Veg,18,989.467222
Classic Kitchen Family Restaurant,19,973.167895
...,...,...
Annapurna Tiffins Punjabi,19,621.828947
Darbar Tiffins Non-Veg,18,596.815556
Darbar Restaurant Punjabi,14,589.972857


In [20]:
df.columns


Index(['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount',
       'restaurant_name_x', 'name', 'city', 'membership', 'restaurant_name_y',
       'cuisine', 'rating', 'rating_range'],
      dtype='str')

In [21]:
df.groupby(["membership", "cuisine"])["total_amount"].sum() \
.sort_values(ascending=False)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [22]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["quarter"] = df["order_date"].dt.to_period("Q")

df.groupby("quarter")["total_amount"].sum() \
.sort_values(ascending=False)


C:\Users\Nethravathi.D\AppData\Local\Temp\ipykernel_43836\565621827.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["order_date"] = pd.to_datetime(df["order_date"])


quarter
2023Q3    2037385.10
2023Q4    2018263.66
2023Q1    1993425.14
2023Q2    1945348.72
2024Q1      17201.50
Freq: Q-DEC, Name: total_amount, dtype: float64

In [27]:
gold_orders_count = df[df['membership'] == 'Gold'].shape[0]
print("Total orders by Gold members:", gold_orders_count)


Total orders by Gold members: 4987


In [28]:
hyderabad_revenue = df[df['city'] == 'Hyderabad']['total_amount'].sum()
print("Total revenue in Hyderabad:", round(hyderabad_revenue))

Total revenue in Hyderabad: 1889367


In [29]:
distinct_users = df['user_id'].nunique()
print("Distinct users who placed orders:", distinct_users)

Distinct users who placed orders: 2883


In [30]:
avg_order_gold = round(df[df['membership'] == 'Gold']['total_amount'].mean(), 2)
print("Average order value for Gold members:", avg_order_gold)

Average order value for Gold members: 797.15


In [31]:
high_rating_orders = df[df['rating'] >= 4.5].shape[0]
print("Orders for restaurants rating >= 4.5:", high_rating_orders)

Orders for restaurants rating >= 4.5: 3374


In [32]:
gold_orders_city = df[df['membership'] == 'Gold'].groupby('city')['total_amount'].sum()
top_city = gold_orders_city.idxmax()
top_city_orders_count = df[(df['membership'] == 'Gold') & (df['city'] == top_city)].shape[0]
print(f"Orders in top revenue city ({top_city}) for Gold members:", top_city_orders_count)

Orders in top revenue city (Chennai) for Gold members: 1337
